In [ ]:
# install on the control webiste librbay code. 

# import subprocess

# import sys



# required_packages = ['img2table']



# for package in required_packages:

#     try:

#         __import__(package)

#     except ImportError:

#         print(f'Installing {package}...')

#         subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])



#------------------------------------------------ Import Lib ----------------------------------------
import re
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pytesseract
import cv2
import numpy as np
from PIL import Image
import base64
import os
import requests
from img2table.ocr import TesseractOCR
from img2table.document import Image

pytesseract.pytesseract.tesseract_cmd =  r"C:\Program Files\Tesseract-OCR\tesseract.exe"

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# %%



#------------------------------------------------ Begin_ fileName ----------------------------------------



regulatorName = 'LC FSRALC' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.0")



now=datetime.datetime.now()



filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])



#scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment



os.chdir(scriptfolder)



tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process







if os.path.exists(tempfolder):



    for rem in os.listdir(tempfolder):



        os.remove(os.path.join(tempfolder, rem))



else:



    os.mkdir(tempfolder)   


# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,
         
		 "profile.default_content_setting_values.automatic_downloads": 1

         
		 }

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


# %%

#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}   

   
regdict={
    
        regulatorName+' 1': 'https://fsrastlucia.org/index.php/insurance-pensions/insurance-companies/regulated-entities', 
        regulatorName+' 2': 'https://fsrastlucia.org/index.php/insurance-pensions/insurance-intermediaries/regulated-entities', 
         regulatorName+' 3': 'https://fsrastlucia.org/index.php/insurance-pensions/pension-fund-plans/regulated-entities', 
         regulatorName+' 4': 'https://fsrastlucia.org/index.php/credit-unions/regulated-entities/credit-unions', 
         regulatorName+' 5': 'https://fsrastlucia.org/index.php/international-sector/international-banks/regulated-entities', 
         regulatorName+' 6': 'https://fsrastlucia.org/index.php/international-sector/international-insurance/regulated-entities', 
         regulatorName+' 7': 'https://fsrastlucia.org/index.php/international-sector/international-mutual-funds/regulated-entities', 
         regulatorName+' 8': 'https://fsrastlucia.org/index.php/money-services-business/regulated-entities', 
          regulatorName+' 9': 'https://fsrastlucia.org/index.php/registered-agents-trustees/regulated-entities', 
        }


Typology={

       regulatorName + ' 1': 'Insurance Companies',
       regulatorName + ' 2': 'Insurance Intermediaries',
       regulatorName + ' 3': 'Registered Pension Fund Plans',
       regulatorName + ' 4': 'Regulated Credit Unions',
       regulatorName + ' 5': 'Licensed International Banks',
       regulatorName + ' 6': 'International Insurance',
       regulatorName + ' 7': 'International Mutual Funds',
       regulatorName + ' 8': 'Money Services Business',
       regulatorName + ' 9': 'Registered Agents & Trustees',
       
        }


processdate = now.strftime('%Y-%m-%d')


# print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict







EMAIL_RE = re.compile(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", re.I)



def normalize_ocr(text):

    s = re.sub(r"\s+", " ", text).strip()

    s = s.replace("Emai", "Email")

    s = s.replace("Email:", "Email: ")

    s = re.sub(r"\s+\.", ".", s)

    return s



def fix_typos(text):

    s = re.sub(r"\blid\b", "ltd", text, flags=re.I)

    return s



def extract_emails(text):

    emails = EMAIL_RE.findall(text)

    cleaned = EMAIL_RE.sub("", text)

    return cleaned, emails



def extract_tel_fax(text):

    tel = ''

    fax = ''

    m = re.search(r"Tel[:.\s]*([0-9\-\/]+)", text, re.I)

    if m:

        tel = m.group(1)

        text = text.replace(m.group(0), "")

    m = re.search(r"Fax[:.\s]*([0-9\-\/]+)", text, re.I)

    if m:

        fax = m.group(1)

        text = text.replace(m.group(0), "")

    return text, tel, fax



def clean_row(text):

    s = normalize_ocr(text)

    s = fix_typos(s)



    s, emails = extract_emails(s)

    s, tel, fax = extract_tel_fax(s)

    s = re.sub(r"Email[:.\s]*", "", s, flags=re.I).strip(" ,;.")



    address = s if s else ''



    return {

        "address": address,

        "tel": tel,

        "fax": fax,

        "emails": emails if emails else ''

    }



def clean_address_only(text):

    s = re.sub(r"\s+", " ", text).strip()

    s = s.replace("Emai", "Email")

    s = re.sub(r"\s+\.", ".", s)



    # remove emails

    s = EMAIL_RE.sub("", s)



    # remove "l:" artifacts and anything after them

    s = re.sub(r"\bl\s*:\s*.*$", "", s, flags=re.I)



    return s.strip(" ,;.")



#------------------------------------------------ Begin_Main ----------------------------------------

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    # print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    rows_before = len(sqldict["Name"])
    if reg == 'LC FSRALC 1' or reg == 'LC FSRALC 2':
        driver.get(regdict[reg])
        img_s = driver.find_elements(By.TAG_NAME,'img')

        sess = requests.Session()
        for c in driver.get_cookies():
            sess.cookies.set(c['name'], c['value'], domain=c.get('domain'))
        sess.headers.update({
            "User-Agent": driver.execute_script("return navigator.userAgent"),
            "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
            "Referer": driver.current_url,
        })

        for index, img in enumerate(img_s):
            if 'arrow' in img.get_attribute('src'):
                continue
            src = img.get_attribute('src') or ''
            if src.startswith("data:image"):
                header, b64 = src.split(",", 1)
                ext = header.split("/")[1].split(";")[0]
                fname = os.path.join(tempfolder, f"image{index}.{ext}")
                with open(fname, "wb") as f:
                    f.write(base64.b64decode(b64))
            elif src.startswith("http"):
                r = sess.get(src, timeout=30,verify=False)
                r.raise_for_status()
                fname = os.path.join(tempfolder, f"image{index}.png")
                with open(fname, "wb") as f:
                    f.write(r.content)
        sleep(3)
        tess_dir = r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\Tesseract-OCR"
        os.environ["PATH"] = tess_dir + ";" + os.environ.get("PATH", "")
        ocr = TesseractOCR(n_threads=1, lang="eng")

        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        for dl_file in dl_files:
            doc = Image(dl_file)
            extracted_tables = doc.extract_tables(ocr=ocr,
                                            implicit_rows=False,
                                            implicit_columns=False,
                                            borderless_tables=False,
                                            min_confidence=50)
            for row in extracted_tables[0].content.values():
                if 'address'  in row[-1].value.replace('\n',' ').lower():
                    continue
                # if not name_cell or 'Insurance ‘Companies' in name_cell:
                #     continue
                # if not addr_cell or 'Address' in addr_cell:
                #     continue

                name_ = row[1].value.replace('\n',' ')
                address_ = row[-1].value.replace('\n',' ')

                cleaned = clean_row(address_)
                address_only = clean_address_only(cleaned['address']) if cleaned['address'] else None
                if not address_only:
                    continue

                tel_ = cleaned['tel'].replace('/','') if cleaned['tel'] else ''
                fax_ = cleaned['fax'] if cleaned['fax'] else ''
                email_ = cleaned['emails'][0] if cleaned['emails'] else ''

                sqldict["Name"].append(name_)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['Address_1'].append(address_only)
                sqldict['Phone'].append(tel_)
                sqldict['Fax'].append(fax_)
                sqldict['Email'].append(email_)
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
        for rem in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, rem))
    
    elif reg == 'LC FSRALC 3' :
        driver.get(regdict[reg])
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'table')))
        table_ = driver.find_element(By.TAG_NAME,'table')
        table_rows = table_.find_elements(By.TAG_NAME,'tr')
        for row in table_rows:
            name_ = row.text
            sqldict["Name"].append(name_)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'LC FSRALC 4' :
        driver.get(regdict[reg])
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'table')))
        table_ = driver.find_element(By.TAG_NAME,'table')
        table_ul = table_.find_element(By.TAG_NAME,'ul')
        table_li = table_.find_elements(By.TAG_NAME,'li')
        for row in table_li:
            name_ = row.text
            #print(name_)
            sqldict["Name"].append(name_)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
    elif reg in ('LC FSRALC 5', 'LC FSRALC 6', 'LC FSRALC 7','LC FSRALC 9'): 
        driver.get(regdict[reg])
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'table')))
        tables_ = driver.find_elements(By.TAG_NAME,'table')
        for table_ in tables_:
            table_rows = table_.find_elements(By.TAG_NAME,'tr')
            for row in table_rows:
                td_cells = row.find_elements(By.TAG_NAME,'td')
                if not td_cells:
                    continue
                name_ = td_cells[0].text
                if 'name'  in name_.lower() :
                    continue
                if any(s in name_.upper() for s in ('DISSOLVED','REVOKED','BANKRUPTCY','BANKRUPT')):
                    continue
                address_ = td_cells[-1].text
                license_no = td_cells[-2].text
                sqldict['Name'].append(name_.replace('\n',' ').strip())
                sqldict['Address_1'].append(address_.replace('\n',' ').strip())
                sqldict['InternalID_1'].append(license_no.replace('\n',' ').strip())
                sqldict['InternalID_1_type'].append('Licence No.')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'LC FSRALC 8' :
        driver.get(regdict[reg])
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'table')))
        tables_ = driver.find_elements(By.TAG_NAME,'table')
        for table_ in tables_:
            table_rows = table_.find_elements(By.TAG_NAME,'tr')
            for row in table_rows:
                td_cells = row.find_elements(By.TAG_NAME,'td')
                if not td_cells:
                    continue
                name_ = td_cells[0].text
                if 'money services business'  in name_.lower() :
                    continue
                address_ = td_cells[-2].text
                #print(name_,address_)
                sqldict['Name'].append(name_.replace('\n',' ').strip())
                sqldict['Address_1'].append(address_.replace('\n',' ').strip())
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
    rows_after = len(sqldict["Name"])
    # print(f"[INFO] : {reg} collected {rows_after - rows_before} rows")

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)

df = df[df['Name']!='']
mask = (df["ListCode"] != "9") | (df["InternalID_1"].fillna("").str.strip() != "")
df = df[mask].copy()
mask = (df["ListCode"].astype(str).str.strip() == "2") & \
       (df["Name"].fillna("").str.lstrip().str.startswith("&"))
df.loc[mask, "Name"] = "Jefftey & Jeffrey Ltd"
df.to_excel(filename, 'SQL Ready', index=False)
driver.quit()
sleep(3)

    
    
    
    